In [2]:
import random
from collections import deque
from typing import List, Tuple, Optional

from wandb.sdk.launch.registry.elastic_container_registry import boto3


def generate_cluster_matrix(
    rows: int,
    cols: int,
    n_clusters: int = 8,
    cluster_size_range: Tuple[int, int] = (20, 120),
    border_cluster_ratio: float = 0.4,
    connectivity: int = 4,  # 4 or 8
    seed: Optional[int] = None,
    allow_overlap: bool = True,
) -> List[List[int]]:
    """
    Generate a 0/1 matrix containing multiple connected clusters of 1s.
    Some clusters are forced to touch the border, others are forced to be interior.

    - connectivity=4 means clusters use N/S/E/W adjacency.
    - connectivity=8 means diagonals also connect.
    """

    if connectivity not in (4, 8):
        raise ValueError("connectivity must be 4 or 8")
    if rows < 3 or cols < 3:
        raise ValueError("rows and cols should be at least 3 to allow interior clusters")

    rng = random.Random(seed)
    grid = [[0 for _ in range(cols)] for _ in range(rows)]

    def neighbors(r: int, c: int):
        steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        if connectivity == 8:
            steps += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
        for dr, dc in steps:
            rr, cc = r + dr, c + dc
            if 0 <= rr < rows and 0 <= cc < cols:
                yield rr, cc

    def is_border(r: int, c: int) -> bool:
        return r == 0 or c == 0 or r == rows - 1 or c == cols - 1

    border_count = max(1, int(round(n_clusters * border_cluster_ratio)))
    interior_count = max(0, n_clusters - border_count)

    def pick_border_seed() -> Tuple[int, int]:
        side = rng.choice(["top", "bottom", "left", "right"])
        if side == "top":
            return (0, rng.randrange(cols))
        if side == "bottom":
            return (rows - 1, rng.randrange(cols))
        if side == "left":
            return (rng.randrange(rows), 0)
        return (rng.randrange(rows), cols - 1)

    def pick_interior_seed(margin: int = 1) -> Tuple[int, int]:
        # margin=1 ensures it's not on the border
        return (rng.randrange(margin, rows - margin), rng.randrange(margin, cols - margin))

    def grow_cluster(seed_cell: Tuple[int, int], target_size: int) -> List[Tuple[int, int]]:
        """
        Grow a connected cluster by random "frontier expansion" from seed_cell.
        Returns list of cells in the cluster.
        """
        sr, sc = seed_cell
        cluster = []
        in_cluster = set()
        frontier = [seed_cell]

        # If overlap not allowed and seed already occupied, bail fast.
        if not allow_overlap and grid[sr][sc] == 1:
            return []

        while frontier and len(cluster) < target_size:
            # Pick a random frontier cell to expand from
            r, c = frontier.pop(rng.randrange(len(frontier)))

            # Skip if overlap not allowed and occupied
            if not allow_overlap and grid[r][c] == 1:
                continue

            if (r, c) in in_cluster:
                continue

            # Add to cluster
            in_cluster.add((r, c))
            cluster.append((r, c))

            # Add neighbors as potential growth sites
            nbrs = list(neighbors(r, c))
            rng.shuffle(nbrs)
            for rr, cc in nbrs:
                if (rr, cc) not in in_cluster:
                    # Keep expanding; if overlap disallowed, don't add occupied cells to frontier
                    if allow_overlap or grid[rr][cc] == 0:
                        frontier.append((rr, cc))

        return cluster

    def paint(cluster_cells: List[Tuple[int, int]]):
        for r, c in cluster_cells:
            grid[r][c] = 1

    # --- Create border-touching clusters ---
    for _ in range(border_count):
        size = rng.randint(*cluster_size_range)
        seed_cell = pick_border_seed()
        cells = grow_cluster(seed_cell, size)
        paint(cells)

    # --- Create interior clusters (attempt to keep them from touching border) ---
    for _ in range(interior_count):
        size = rng.randint(*cluster_size_range)

        # We'll try multiple times to find a seed/shape that stays interior.
        # If overlap is allowed or matrix is crowded, this is still usually fine.
        best_cells = []
        for _attempt in range(30):
            seed_cell = pick_interior_seed(margin=1)
            cells = grow_cluster(seed_cell, size)

            # Enforce interior: no cell may be on the border
            if cells and all(not is_border(r, c) for r, c in cells):
                best_cells = cells
                break

            # Keep the "most interior" attempt as fallback
            if len(cells) > len(best_cells) and cells:
                # score = number of border cells (lower is better)
                best_cells = cells

        # If fallback cluster touches border, prune border cells while keeping connectivity roughly.
        # Simple pruning: remove border cells from the set before painting.
        if best_cells and any(is_border(r, c) for r, c in best_cells):
            best_cells = [(r, c) for (r, c) in best_cells if not is_border(r, c)]

        paint(best_cells)

    return grid


def print_matrix(mat: List[List[int]]):
    for row in mat:
        print("".join(str(v) for v in row))


m = generate_cluster_matrix(
	rows=35,
	cols=80,
	n_clusters=10,
	cluster_size_range=(30, 180),
	border_cluster_ratio=0.4,
	connectivity=4,
	seed=7,
	allow_overlap=True,
)
print_matrix(m)

00000000000000000000000000000000000000000000000000000000000000000000000001101111
00000000000000000000000000000000000000000000000000000000000000000000000000111111
00000000000000000000000000000000000000000000000000000000000000000000111100101111
00000000000000000000000000000000000000000000000000000000000000000000111000111111
00000000000000000000000000000000000000000000000000000000000000000010111111111111
00000000000000000000000000000000000000000000000000000000000000000111111111111111
00000000000000000000000000000000000000000000000000000000000000000010111111111111
00000000000000000000000000000000000000000000000000000000000000000001111111111111
00000000000000000000000000000000000000000000000000000000000000000000111111111111
00000000000000000000000100000000000000000000000000000000000000000001111111111111
00000000000001000000000111110000000000000000000000000000000000000000111111111111
00000000000001110001111111000000000000000000000000000000000000011111111111111111
0000000000001111100111111100

In [10]:
dp_matrix = [[-1 for j in range(len(m[i]))] for i in range(len(m))]

In [14]:
def is_island_recursive(i, j, previous_i, previous_j):
	if dp_matrix[i][j] != -1:
		return dp_matrix[i][j]

	if m[i][j] == 0:
		dp_matrix[i][j] = False
		return False

	left_idx = i, j-1
	right_idx = i, j+1
	top_idx = i-1, j
	bot_idx = i+1, j

	for idx in [left_idx, right_idx, top_idx, bot_idx]:
		try:
			m[idx[0]][idx[1]]
		except IndexError:
			dp_matrix[i][j] = False
			return False

	for idx in [left_idx, right_idx, top_idx, bot_idx]:
		if (idx[0], idx[1]) != (previous_i, previous_j):
			if is_island_recursive(idx[0], idx[1], i, j):
				dp_matrix[idx[0]][idx[1]] = True
				return True
	return False

for i in range(len(m)):
	for j in range(len(m[0])):
		if is_island_recursive(i, j, i, j):
			m[i][j] = 0

print_matrix(m)

RecursionError: maximum recursion depth exceeded